# 1 · Create the Vector Search Index

Databricks **Free Edition does not include Knowledge Assistant**, so this
notebook builds the document-retrieval backend with **Vector Search** instead.

The contract PDFs are **print-to-PDF documents whose body is image-based**, so
plain text extraction returns only page headers. We use Databricks' managed
document parsing — **`ai_parse_document`** (the Information Extraction
capability) — to OCR the full contract text, including tables and terms.

**What this notebook does**
1. Parses every contract PDF with `ai_parse_document` (OCR + layout)
2. Flattens the parsed elements into clean text and chunks it into a Delta table
3. Creates a Vector Search endpoint
4. Creates a **Delta Sync index** with managed embeddings (`databricks-gte-large-en`)

The index is attached directly to the Supervisor Agent in notebook `3` as a
native **Vector Search index** sub-agent.

### Define Parameters
Review the widgets before running. Defaults match the **Marketing Campaign Analytics** layout.
| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `marketing_campaign_analytics` | Unity Catalog **catalog** for all demo objects (tables, volumes, indexes). |
| `SCHEMA_NAME` | `marketing_campaign` | Schema inside the catalog for sales data and contracts. |
| `VOLUME_NAME` | `contracts` | Volume containing the fictional contract PDFs. |
| `PARSED_TABLE` | `contract_text` | Staging Delta table for OCR output from `ai_parse_document`. |
| `SOURCE_TABLE` | `contract_chunks` | Chunked text table (CDF enabled) synced to Vector Search. |
| `VS_ENDPOINT_NAME` | `marketing_campaign_vs` | Vector Search endpoint name. |
| `INDEX_NAME` | `contract_chunks_index` | Vector Search index name (also the UC securable name). |
| `EMBEDDING_MODEL` | `databricks-gte-large-en` | Embedding model endpoint for managed embeddings. |
| `CHUNK_SIZE` | `1200` | Characters per text chunk. |
| `CHUNK_OVERLAP` | `150` | Overlap between consecutive chunks. |


In [ ]:
dbutils.widgets.text("CATALOG_NAME", "marketing_campaign_analytics", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
dbutils.widgets.text("VOLUME_NAME", "contracts", "VOLUME_NAME — UC volume with fictional_*.pdf contracts")
dbutils.widgets.text("PARSED_TABLE", "contract_text", "PARSED_TABLE — Delta table for ai_parse_document output")
dbutils.widgets.text("SOURCE_TABLE", "contract_chunks", "SOURCE_TABLE — chunked text table (CDF enabled)")
dbutils.widgets.text("VS_ENDPOINT_NAME", "marketing_campaign_vs", "VS_ENDPOINT_NAME — Vector Search endpoint name")
dbutils.widgets.text("INDEX_NAME", "contract_chunks_index", "INDEX_NAME — Vector Search index (from notebook 1)")
dbutils.widgets.text("EMBEDDING_MODEL", "databricks-gte-large-en", "EMBEDDING_MODEL — managed embedding model endpoint")
dbutils.widgets.text("CHUNK_SIZE", "1200", "CHUNK_SIZE — characters per chunk")
dbutils.widgets.text("CHUNK_OVERLAP", "150", "CHUNK_OVERLAP — overlap between chunks")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
PARSED_TABLE = dbutils.widgets.get("PARSED_TABLE")
SOURCE_TABLE = dbutils.widgets.get("SOURCE_TABLE")
VS_ENDPOINT_NAME = dbutils.widgets.get("VS_ENDPOINT_NAME")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
EMBEDDING_MODEL = dbutils.widgets.get("EMBEDDING_MODEL")
CHUNK_SIZE = int(dbutils.widgets.get("CHUNK_SIZE"))
CHUNK_OVERLAP = int(dbutils.widgets.get("CHUNK_OVERLAP"))

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
PARSED_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{PARSED_TABLE}"
SOURCE_TABLE_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{SOURCE_TABLE}"
INDEX_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"
print("Contracts volume :", VOLUME_PATH)
print("Parsed table     :", PARSED_FQN)
print("Chunks table     :", SOURCE_TABLE_FQN)
print("Index            :", INDEX_FQN)

### Parse the Contract PDFs with `ai_parse_document`

`ai_parse_document` runs managed OCR + layout analysis on each PDF. We flatten
the returned elements into a single text column per document, dropping page
headers and footers (the print boilerplate).

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}')

spark.sql(f'''
CREATE OR REPLACE TABLE {PARSED_FQN} AS
WITH parsed AS (
  SELECT regexp_extract(path, '([^/]+)$', 1) AS file_name,
         ai_parse_document(content) AS doc
  FROM READ_FILES('{VOLUME_PATH}/', format => 'binaryFile')
)
SELECT file_name,
       concat_ws(' ', transform(
         filter(
           from_json(to_json(doc:document:elements), 'array<struct<content:string,type:string>>'),
           e -> e.type NOT IN ('page_header', 'page_footer')
         ),
         e -> e.content)) AS content
FROM parsed
''')

cnt = spark.table(PARSED_FQN).count()
print(f'Parsed {cnt} contract documents.')
display(spark.sql(f'SELECT file_name, length(content) AS chars FROM {PARSED_FQN} ORDER BY chars DESC LIMIT 5'))

### Chunk the Parsed Text

The source table for a Delta Sync index must have **Change Data Feed** enabled.

In [ ]:
def chunk_text(text, size, overlap):
    text = ' '.join((text or '').split())
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return [c for c in chunks if c.strip()]

rows = []
for r in spark.table(PARSED_FQN).select('file_name', 'content').collect():
    for i, chunk in enumerate(chunk_text(r['content'], CHUNK_SIZE, CHUNK_OVERLAP)):
        rows.append({'id': f'{r["file_name"]}::{i}', 'file_name': r['file_name'], 'chunk_index': i, 'content': chunk})

from pyspark.sql import Row
df = spark.createDataFrame([Row(**x) for x in rows])
(df.write.format('delta').mode('overwrite')
   .option('delta.enableChangeDataFeed', 'true')
   .option('overwriteSchema', 'true')
   .saveAsTable(SOURCE_TABLE_FQN))
spark.sql(f'ALTER TABLE {SOURCE_TABLE_FQN} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')
print(f'Wrote {df.count()} chunks to {SOURCE_TABLE_FQN}')

### Create the Vector Search Endpoint

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType, VectorIndexType, PipelineType,
    DeltaSyncVectorIndexSpecRequest, EmbeddingSourceColumn,
)

w = WorkspaceClient()

existing = [e.name for e in w.vector_search_endpoints.list_endpoints()]
if VS_ENDPOINT_NAME in existing:
    print('Reusing endpoint', VS_ENDPOINT_NAME)
else:
    print('Creating endpoint', VS_ENDPOINT_NAME, '(may take a few minutes)...')
    w.vector_search_endpoints.create_endpoint_and_wait(name=VS_ENDPOINT_NAME, endpoint_type=EndpointType.STANDARD)
    print('Endpoint online.')

### Create (or Sync) the Delta Sync Index

In [ ]:
import time

def index_exists(fqn):
    try:
        w.vector_search_indexes.get_index(index_name=fqn); return True
    except Exception:
        return False

if index_exists(INDEX_FQN):
    print('Index exists — triggering sync')
    w.vector_search_indexes.sync_index(index_name=INDEX_FQN)
else:
    w.vector_search_indexes.create_index(
        name=INDEX_FQN, endpoint_name=VS_ENDPOINT_NAME, primary_key='id',
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=SOURCE_TABLE_FQN, pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[EmbeddingSourceColumn(name='content', embedding_model_endpoint_name=EMBEDDING_MODEL)],
        ),
    )
    print('Created index', INDEX_FQN)

### Wait for the Index to Finish its Sync

In [ ]:
deadline = time.time() + 60 * 25
while True:
    idx = w.vector_search_indexes.get_index(index_name=INDEX_FQN)
    ready = getattr(idx.status, 'ready', None)
    rows_indexed = getattr(idx.status, 'indexed_row_count', None)
    print(f'ready={ready} indexed_rows={rows_indexed}')
    if ready:
        break
    if time.time() > deadline:
        print('Still syncing; it is safe to continue once ready=True.'); break
    time.sleep(30)

### Result

In [ ]:
print('Vector Search endpoint :', VS_ENDPOINT_NAME)
print('Vector Search index    :', INDEX_FQN)
print('Embedding model        :', EMBEDDING_MODEL)